In [2]:
import json
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score, precision_score, recall_score
from typing import Optional
from huggingface_hub import hf_hub_download

In [3]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def macro_metrics(
    y_true,
    y_pred,
    labels=(-1, 0, 1)
):
    """
    Macro class-wise metrics:
    - Compute per-aspect multiclass F1/Precision/Recall
    - Average across aspects

    y_true, y_pred: (N, S)
    """

    precisions, recalls, f1s = [], [], []

    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        precisions.append(
            precision_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        recalls.append(
            recall_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        f1s.append(
            f1_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )

    return {
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
    }
def sample_metrics(
    y_true,
    y_pred,
    labels=(-1, 0, 1)
):
    """
    Sample-based metrics:
    - Compute metrics per sample across all aspects
    - Then average over samples
    """

    precisions, recalls, f1s = [], [], []

    for i in range(y_true.shape[0]):
        yt = y_true[i]
        yp = y_pred[i]

        precisions.append(
            precision_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        recalls.append(
            recall_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )
        f1s.append(
            f1_score(
                yt, yp,
                labels=list(labels),
                average="macro",
                zero_division=0
            )
        )

    return {
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
    }


In [4]:
class EvalDataset(Dataset):
    def __init__(self, data, tokenizer, top_dim, sub_dim, max_len=256, label_map=None):
        self.data = data
        self.tokenizer = tokenizer
        self.top_dim = top_dim
        self.sub_dim = sub_dim
        self.max_len = max_len
        self.label_map = label_map # Store label_map

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["text"]

        # ----------------------------
        # Tokenize text
        # ----------------------------
        enc = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}

        # ----------------------------
        # Top-level labels (multi-hot)
        # ----------------------------
        top = torch.tensor(item["top_cluster_ids"], dtype=torch.float32)
        if top.ndim == 0:  # single int label
            top = F.one_hot(top.long(), num_classes=self.top_dim).float()

        # ----------------------------
        # Sub-level labels (robust binary creation)
        # ----------------------------
        sub_ids = torch.tensor(item["sub_cluster_ids"], dtype=torch.float32)

        # ----------------------------
        raw_sentiments = item.get("sentiments", {})
        sentiments = {}
        for k, v in raw_sentiments.items():
            mapped_v = v
            if self.label_map:
                mapped_v = self.label_map.get(v)

            if mapped_v is not None: # Only include if a valid (non-None) value is found/mapped
                sentiments[int(k)] = mapped_v

        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "top_labels": top,
            "sub_labels": sub_ids,
            "sentiments": sentiments,
        }

    @staticmethod
    def collate_fn(batch):
        return {
            "input_ids": torch.stack([b["input_ids"] for b in batch]),
            "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
            "top_labels": torch.stack([b["top_labels"] for b in batch]),
            "sub_labels": torch.stack([b["sub_labels"] for b in batch]),
            "sentiments": [b["sentiments"] for b in batch],
        }

In [5]:
def load_tokenizer_and_model_from_hf():
    repo_id = "Faisal191/aspect-classifier"
    encoder_subfolder = "Domain_trained_encoder"

    onnx_file = "HABSA/Habsa_v6_fp32.onnx"

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        repo_id,
        subfolder=encoder_subfolder)
    # Download ONNX graph
    onnx_path = hf_hub_download(
        repo_id=repo_id,
        filename=onnx_file,
        repo_type="model")

    return tokenizer, onnx_path

tokenizer, onnx_path = load_tokenizer_and_model_from_hf()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

Domain_trained_encoder/spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

HABSA/Habsa_v6_fp32.onnx:   0%|          | 0.00/811M [00:00<?, ?B/s]

In [6]:
top_logits, sub_logits, sent_logits = [], [], []

In [8]:
top_logits = np.load("/content/top_logits.npy")
sub_logits = np.load("/content/sub_logits.npy")
sent_logits = np.load("/content/sent_logits.npy")

In [9]:
print(top_logits.shape)
print(sub_logits.shape)
print(sent_logits.shape)

(793, 9)
(793, 29)
(793, 29, 2)


In [10]:
outputs = {"top_logits": top_logits,
           "sub_logits": sub_logits,
           "sent_logits": sent_logits}

In [11]:
def sentiment_from_logits(sent_logits, pos_margin, neg_margin):
    delta = sent_logits[..., 1] - sent_logits[..., 0]

    sent = torch.full_like(delta, -1, dtype=torch.long)  # -1 = abstain
    sent[delta > pos_margin] = 1
    sent[delta < -neg_margin] = 0
    return sent


In [12]:
def inference(top_to_sub_dense, top_logits, sub_logits, sent_logits, top_thresholds,
              sub_thresholds, pos_margin, neg_margin):
    top_logits = torch.from_numpy(top_logits)
    sub_logits = torch.from_numpy(sub_logits)
    sent_logits = torch.from_numpy(sent_logits)
    p_top = torch.sigmoid(top_logits)        # (B, T)
    p_sub = torch.sigmoid(sub_logits)        # (B, S)
    top_thr = torch.tensor(top_thresholds).unsqueeze(0)   # (1, T)
    sub_thr = torch.tensor(sub_thresholds).unsqueeze(0)   # (1, S)
    p_top_bin = (p_top > top_thr).to(torch.float32)
    top_to_sub_dense = torch.tensor(top_to_sub_dense)
    mask = torch.matmul(p_top_bin, top_to_sub_dense)  # (B, S)
    mask = mask.clamp(0.0, 1.0)
    no_top = (p_top_bin.sum(dim=1, keepdim=True) == 0).to(torch.float32)  # (B, 1)
    mask = mask + no_top * (1.0 - mask)
    p_sub_masked = p_sub * mask
    pred_sub_bin = (p_sub_masked > sub_thr).to(torch.float32)
    sent_preds = sentiment_from_logits(sent_logits, pos_margin, neg_margin)

    return pred_sub_bin.cpu().numpy(), sent_preds.cpu().numpy()

In [13]:
def eval(pred_sub_bin, sent_preds, y_sub, sent_labels):
    f1_sub = f1_score(y_sub, pred_sub_bin, average="macro", zero_division=0)
    recall_sub = recall_score(y_sub, pred_sub_bin, average="macro", zero_division=0)
    precision_sub = precision_score(y_sub, pred_sub_bin, average="macro", zero_division=0)

    sent_preds_flat = sent_preds.flatten()
    sent_labels_flat = sent_labels.flatten()

    mask = sent_labels_flat != -1
    f1_sent = f1_score(sent_labels_flat[mask], sent_preds_flat[mask], average="macro", zero_division=0)
    rec_sent = recall_score(sent_labels_flat[mask], sent_preds_flat[mask], average="macro", zero_division=0)
    prec_sent = precision_score(sent_labels_flat[mask], sent_preds_flat[mask], average="macro", zero_division=0)
    joint_preds = sent_preds.copy()
    joint_preds[(pred_sub_bin == 0)] = -1
    hard_class = macro_metrics(sent_labels, joint_preds)

    return {"joint_precision": hard_class["precision"], "joint_recall": hard_class["recall"],
            "joint_f1": hard_class["f1"], "sentiment_f1": f1_sent, "sentiment_precision": prec_sent, "sentiment_recall": rec_sent ,
            "sub_f1": f1_sub, "aspect_precision": precision_sub, "aspect_recall": recall_sub}



In [14]:
CFG = {
    "hierarchical_json": r"/content/final_aspa_data_hierarchical_with_sentiments_temp_v6.json",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


In [15]:
def run(cfg):
    DEVICE = torch.device(cfg["device"])
    sample_json = json.load(open(cfg["hierarchical_json"], "r"))
    eval_dataset = EvalDataset(sample_json, tokenizer, top_dim=9, sub_dim=29, max_len=256, label_map={0: 0, 1: 1})
    valid_indices = np.load("/content/valid_indices (2).npy")
    top_to_sub_dense = np.load("/content/top_to_sub_dense (1).npy")
    eval_ds = torch.utils.data.Subset(eval_dataset, valid_indices)
    eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False, collate_fn=EvalDataset.collate_fn)

    y_sub, sentiments = [], []
    for batch in tqdm(eval_loader):
        y_sub.append(batch["sub_labels"].cpu().numpy())
        sentiments += batch["sentiments"]
    y_sub = np.concatenate(y_sub)

    len_eval_data = len(eval_loader.dataset)
    sent_labels = np.full((len_eval_data, 29), -1.0)

    for i, row_i in enumerate(sentiments):
      aspect, sentiments = list(row_i.keys()), list(row_i.values())
      sent_labels[i, aspect] = sentiments
    return y_sub, sent_labels, top_to_sub_dense

In [17]:
 y_sub, sent_labels, top_to_sub_dense = run(CFG)

100%|██████████| 13/13 [00:00<00:00, 13.11it/s]


In [18]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 14.3 MB/s eta 0:00:00


In [19]:
import optuna

In [28]:
FIXED_TOP_THRESHOLDS = [0.5] * 9

In [29]:
study = optuna.create_study(
    directions=[
        "maximize",  # joint precision
        "maximize",  # joint recall
        "maximize",  # aspect precision
        "maximize",  # aspect recall
        "maximize",  # sentiment precision
        "maximize",  # sentiment recall
    ],
    sampler=optuna.samplers.TPESampler(seed=42)
)

def optuna_objective(trial):
    if trial.number < 800:
        top_thresholds = FIXED_TOP_THRESHOLDS
    else:
        top_thresholds = [
            trial.suggest_float(f"top_thr_{j}", 0.4, 0.9)
            for j in range(9)
        ]

    sub_thresholds = [
        trial.suggest_float(f"sub_thr_{j}", 0.4, 0.9)
        for j in range(29)
    ]

    pos_margin = trial.suggest_float("pos_margin", 0.0, 2.0)
    neg_margin = trial.suggest_float("neg_margin", 0.0, 1.5)

    pred_sub_bin, sent_preds = inference(
        top_to_sub_dense,
        outputs["top_logits"],
        outputs["sub_logits"],
        outputs["sent_logits"],
        top_thresholds,
        sub_thresholds,
        pos_margin,
        neg_margin)

    metrics = eval(pred_sub_bin, sent_preds, y_sub, sent_labels)

    return (
        metrics["joint_precision"],
        metrics["joint_recall"],
        metrics["aspect_precision"],
        metrics["aspect_recall"],
        metrics["sentiment_precision"],
        metrics["sentiment_recall"],
    )


[I 2025-12-28 21:43:27,650] A new study created in memory with name: no-name-5d2c93e9-8baa-4bed-bdc2-721b3afc2f6b


In [30]:
study.optimize(optuna_objective, n_trials=2000)

[I 2025-12-28 21:43:34,514] Trial 0 finished with values: [0.6707375431199077, 0.6011304750248866, 0.7557334085396777, 0.6282825832808153, 0.515414197060858, 0.4436426744610888] and parameters: {'sub_thr_0': 0.5872700594236813, 'sub_thr_1': 0.8753571532049581, 'sub_thr_2': 0.7659969709057026, 'sub_thr_3': 0.6993292420985183, 'sub_thr_4': 0.4780093202212183, 'sub_thr_5': 0.47799726016810135, 'sub_thr_6': 0.42904180608409975, 'sub_thr_7': 0.8330880728874677, 'sub_thr_8': 0.7005575058716045, 'sub_thr_9': 0.7540362888980228, 'sub_thr_10': 0.41029224714790125, 'sub_thr_11': 0.8849549260809972, 'sub_thr_12': 0.8162213204002109, 'sub_thr_13': 0.5061695553391381, 'sub_thr_14': 0.49091248360355033, 'sub_thr_15': 0.49170225492671693, 'sub_thr_16': 0.5521211214797689, 'sub_thr_17': 0.6623782158161189, 'sub_thr_18': 0.615972509321058, 'sub_thr_19': 0.5456145700990209, 'sub_thr_20': 0.7059264473611897, 'sub_thr_21': 0.46974693032602094, 'sub_thr_22': 0.5460723242676091, 'sub_thr_23': 0.583180921646

In [31]:
pareto_trials = study.best_trials

In [35]:
len(pareto_trials)

1111

In [42]:
import pandas as pd
from google.colab import files  # Only if you want to auto-download

# 1. Convert Study to DataFrame
df = study.trials_dataframe()

# 2. Filter for only COMPLETE trials (ignore failed/pruned ones)
df = df[df["state"] == "COMPLETE"]

# 3. Rename columns for sanity
# Check your optuna objective to see which is 0 and which is 1
# Example: return metrics["joint_f1"], metrics["joint_recall"]
df = df.rename(columns={
    "values_0": "joint_precision",
    "values_1": "joint_recall",
    "values_2": "aspect_precision",
    "values_3": "aspect_recall",
    "values_4": "sentiment_precision",
    "values_5": "sentiment_recall",
})

# 2. Get the list of IDs for the Pareto efficient trials
best_trial_numbers = [t.number for t in study.best_trials]

# 3. Filter the dataframe using .isin()
pareto_df = df[df['number'].isin(best_trial_numbers)]

# 4. Save to CSV
csv_filename = "optuna_results_1111_trials.csv"
pareto_df.to_csv(csv_filename, index=False)

print(f"✅ Saved {len(df)} trials to {csv_filename}")

# 5. Download to your local machine (Optional)
files.download(csv_filename)

      number  joint_precision  joint_recall  aspect_precision  aspect_recall  \
2          2         0.695454      0.552731          0.773222       0.620816   
11        11         0.716257      0.543023          0.771593       0.615218   
22        22         0.664453      0.594852          0.754443       0.636213   
25        25         0.660358      0.576307          0.742636       0.651795   
32        32         0.669459      0.587457          0.757211       0.638678   
...      ...              ...           ...               ...            ...   
1993    1993         0.710384      0.552238          0.766216       0.665717   
1995    1995         0.669461      0.635560          0.768431       0.655374   
1996    1996         0.686664      0.603448          0.767885       0.660842   
1997    1997         0.716004      0.558618          0.769972       0.657399   
1998    1998         0.669298      0.631162          0.769257       0.659994   

      sentiment_precision  sentiment_re

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
pareto_df

,number,joint_precision,joint_recall,aspect_precision,aspect_recall,sentiment_precision,sentiment_recall,datetime_start,datetime_complete,duration,...,params_top_thr_0,params_top_thr_1,params_top_thr_2,params_top_thr_3,params_top_thr_4,params_top_thr_5,params_top_thr_6,params_top_thr_7,params_top_thr_8,state
2,2,0.695454,0.552731,0.773222,0.620816,0.563986,0.351209,2025-12-28 21:43:34.987294,2025-12-28 21:43:35.388510,0 days 00:00:00.401216,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
11,11,0.716257,0.543023,0.771593,0.615218,0.576409,0.336028,2025-12-28 21:43:38.266204,2025-12-28 21:43:38.546233,0 days 00:00:00.280029,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
22,22,0.664453,0.594852,0.754443,0.636213,0.524461,0.437313,2025-12-28 21:43:41.326881,2025-12-28 21:43:41.625370,0 days 00:00:00.298489,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
25,25,0.660358,0.576307,0.742636,0.651795,0.542349,0.387212,2025-12-28 21:43:42.205513,2025-12-28 21:43:42.487717,0 days 00:00:00.282204,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
32,32,0.669459,0.587457,0.757211,0.638678,0.536313,0.408359,2025-12-28 21:43:44.238433,2025-12-28 21:43:44.545824,0 days 00:00:00.307391,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1993,1993,0.710384,0.552238,0.766216,0.665717,0.582661,0.325075,2025-12-28 23:57:06.453442,2025-12-28 23:57:12.613524,0 days 00:00:06.160082,...,0.720347,0.458494,0.778110,0.691220,0.414180,0.772289,0.673633,0.598735,0.493876,COMPLETE
1995,1995,0.669461,0.635560,0.768431,0.655374,0.490639,0.478865,2025-12-28 23:57:18.423264,2025-12-28 23:57:24.566617,0 days 00:00:06.143353,...,0.823715,0.449997,0.476960,0.677186,0.707567,0.763908,0.694722,0.615685,0.476007,COMPLETE
1996,1996,0.686664,0.603448,0.767885,0.660842,0.528127,0.420556,2025-12-28 23:57:24.567600,2025-12-28 23:57:30.493380,0 days 00:00:05.925780,...,0.697598,0.462799,0.766225,0.684313,0.443487,0.743631,0.701713,0.584832,0.452992,COMPLETE
1997,1997,0.716004,0.558618,0.769972,0.657399,0.570235,0.339997,2025-12-28 23:57:30.494444,2025-12-28 23:57:37.340916,0 days 00:00:06.846472,...,0.855524,0.431617,0.499351,0.696769,0.463125,0.790340,0.715324,0.609503,0.443541,COMPLETE
